# MGMT298D: Science and Strategy of AI## Week 8: LLM Parameters & Generative AI### UCLA Anderson School of Management

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom scipy.special import softmaxfrom collections import defaultdictnp.random.seed(42)sns.set_style('whitegrid')%matplotlib inline

## 1. Simulated Vocabulary & Base Distribution

In [ ]:
# Define token vocabulary with base probabilities (skewed towards positive tokens)vocabulary = ["great", "good", "okay", "fine", "excellent",               "wonderful", "amazing", "decent", "poor", "terrible"]base_probs = np.array([0.15, 0.12, 0.10, 0.08, 0.18, 0.16, 0.14, 0.05, 0.01, 0.01])base_probs = base_probs / base_probs.sum()  # Normalize to sum to 1print("Token Vocabulary:")print(vocabulary)print("\nBase Probability Distribution:")for token, prob in zip(vocabulary, base_probs):    print(f"  {token:12s}: {prob:.4f}")

## 2. Temperature Scaling

In [ ]:
# Apply temperature scaling: lower T → more peaked, higher T → more flatdef apply_temperature(probs, temperature):    logits = np.log(probs + 1e-10)    scaled_logits = logits / temperature    return softmax(scaled_logits)temperatures = [0.2, 0.5, 1.0, 1.5, 2.0]temp_distributions = {t: apply_temperature(base_probs, t) for t in temperatures}print("Temperature Scaling Results:")for temp in temperatures:    print(f"\nTemperature = {temp}:")    top_3_idx = np.argsort(temp_distributions[temp])[-3:][::-1]    for idx in top_3_idx:        print(f"  {vocabulary[idx]:12s}: {temp_distributions[temp][idx]:.4f}")

In [ ]:
# Visualize temperature effect on distributionfig, ax = plt.subplots(figsize=(12, 6))x = np.arange(len(vocabulary))width = 0.15for i, temp in enumerate(temperatures):    offset = (i - 2) * width    ax.bar(x + offset, temp_distributions[temp], width, label=f'T={temp}', alpha=0.8)ax.set_xlabel('Token', fontsize=11, fontweight='bold')ax.set_ylabel('Probability', fontsize=11, fontweight='bold')ax.set_title('Effect of Temperature on Probability Distribution', fontsize=12, fontweight='bold')ax.set_xticks(x)ax.set_xticklabels(vocabulary, rotation=45, ha='right')ax.legend(fontsize=10)ax.grid(alpha=0.3)plt.tight_layout()plt.show()

## 3. Temperature Sampling Demo

In [ ]:
# Sample 100 tokens from each temperature distributionnum_samples = 100sampled_tokens = {}for temp in temperatures:    samples = np.random.choice(len(vocabulary), size=num_samples,                               p=temp_distributions[temp])    sampled_tokens[temp] = [vocabulary[i] for i in samples]# Count frequenciesfreq_data = {}for temp in temperatures:    freq_data[temp] = {token: sampled_tokens[temp].count(token)                        for token in vocabulary}print("Token Frequencies (100 samples each):")for temp in temperatures:    print(f"\nTemperature = {temp}:")    sorted_freq = sorted(freq_data[temp].items(), key=lambda x: x[1], reverse=True)    for token, count in sorted_freq[:5]:        print(f"  {token:12s}: {count:2d} ({count/num_samples:.1%})")

In [ ]:
# Visualize sampling frequencies across temperaturesfig, ax = plt.subplots(figsize=(12, 5))x = np.arange(len(vocabulary))width = 0.15for i, temp in enumerate(temperatures):    offset = (i - 2) * width    freqs = [freq_data[temp][token] for token in vocabulary]    ax.bar(x + offset, freqs, width, label=f'T={temp}', alpha=0.8)ax.set_xlabel('Token', fontsize=11, fontweight='bold')ax.set_ylabel('Sample Count (out of 100)', fontsize=11, fontweight='bold')ax.set_title('Token Frequencies: Temperature Sampling (100 samples each)',              fontsize=12, fontweight='bold')ax.set_xticks(x)ax.set_xticklabels(vocabulary, rotation=45, ha='right')ax.legend(fontsize=10)ax.grid(alpha=0.3)plt.tight_layout()plt.show()

## 4. Top-K Filtering

In [ ]:
# Keep only top-k tokens, zero out rest, renormalizedef apply_top_k(probs, k):    filtered = probs.copy()    top_k_idx = np.argsort(filtered)[-k:]    mask = np.zeros_like(filtered, dtype=bool)    mask[top_k_idx] = True    filtered[~mask] = 0    return filtered / filtered.sum()k_values = [3, 5, 10]topk_distributions = {k: apply_top_k(base_probs, k) for k in k_values}print("Top-K Filtering Results:")for k in k_values:    print(f"\nTop-K = {k}:")    nonzero_idx = np.where(topk_distributions[k] > 0)[0]    for idx in nonzero_idx:        print(f"  {vocabulary[idx]:12s}: {topk_distributions[k][idx]:.4f}")

In [ ]:
# Visualize Top-K effectfig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)for idx, k in enumerate(k_values):    ax = axes[idx]    colors = ['#1f77b4' if topk_distributions[k][i] > 0 else '#d3d3d3'               for i in range(len(vocabulary))]    ax.bar(range(len(vocabulary)), topk_distributions[k], color=colors, alpha=0.8)    ax.set_xlabel('Token', fontsize=10)    if idx == 0:        ax.set_ylabel('Probability', fontsize=10)    ax.set_title(f'Top-K = {k}', fontsize=11, fontweight='bold')    ax.set_xticks(range(len(vocabulary)))    ax.set_xticklabels(vocabulary, rotation=45, ha='right', fontsize=9)    ax.grid(alpha=0.3)plt.tight_layout()plt.show()

## 5. Top-P (Nucleus) Sampling

In [ ]:
# Keep tokens until cumulative probability exceeds pdef apply_top_p(probs, p):    sorted_idx = np.argsort(probs)[::-1]    cumsum = np.cumsum(probs[sorted_idx])    cutoff_idx = np.where(cumsum <= p)[0]        filtered = np.zeros_like(probs)    filtered[sorted_idx[cutoff_idx]] = probs[sorted_idx[cutoff_idx]]    return filtered / filtered.sum()p_values = [0.3, 0.5, 0.9]topp_distributions = {p: apply_top_p(base_probs, p) for p in p_values}print("Top-P Filtering Results:")for p in p_values:    nonzero_count = np.sum(topp_distributions[p] > 0)    print(f"\nTop-P = {p} (keeps {nonzero_count} tokens):")    nonzero_idx = np.where(topp_distributions[p] > 0)[0]    for idx in nonzero_idx:        print(f"  {vocabulary[idx]:12s}: {topp_distributions[p][idx]:.4f}")

In [ ]:
# Visualize Top-P effectfig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)for idx, p in enumerate(p_values):    ax = axes[idx]    colors = ['#2ca02c' if topp_distributions[p][i] > 0 else '#d3d3d3'               for i in range(len(vocabulary))]    ax.bar(range(len(vocabulary)), topp_distributions[p], color=colors, alpha=0.8)    ax.set_xlabel('Token', fontsize=10)    if idx == 0:        ax.set_ylabel('Probability', fontsize=10)    ax.set_title(f'Top-P = {p}', fontsize=11, fontweight='bold')    ax.set_xticks(range(len(vocabulary)))    ax.set_xticklabels(vocabulary, rotation=45, ha='right', fontsize=9)    ax.grid(alpha=0.3)plt.tight_layout()plt.show()

## 6. Combined Effects (Temperature + Top-K + Top-P)

In [ ]:
# Apply all three techniques in sequencedef apply_all_constraints(probs, temperature, k, p):    # Step 1: Temperature    result = apply_temperature(probs, temperature)    # Step 2: Top-K    result = apply_top_k(result, k)    # Step 3: Top-P    result = apply_top_p(result, p)    return resultsettings = {    'Conservative': {'T': 0.3, 'K': 5, 'P': 0.5},    'Balanced': {'T': 0.7, 'K': 20, 'P': 0.9},    'Creative': {'T': 1.5, 'K': 50, 'P': 0.95}}combined_distributions = {}for name, params in settings.items():    combined_distributions[name] = apply_all_constraints(        base_probs, params['T'], params['K'], params['P']    )print("Combined Parameter Effects:")for name, dist in combined_distributions.items():    params = settings[name]    print(f"\n{name} (T={params['T']}, K={params['K']}, P={params['P']}):")    top_5_idx = np.argsort(dist)[-5:][::-1]    for idx in top_5_idx:        if dist[idx] > 0:            print(f"  {vocabulary[idx]:12s}: {dist[idx]:.4f}")

In [ ]:
# Visualize combined effectsfig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)names = list(settings.keys())for idx, name in enumerate(names):    ax = axes[idx]    dist = combined_distributions[name]    colors = ['#ff7f0e' if dist[i] > 0 else '#d3d3d3' for i in range(len(vocabulary))]    ax.bar(range(len(vocabulary)), dist, color=colors, alpha=0.8)    ax.set_xlabel('Token', fontsize=10)    if idx == 0:        ax.set_ylabel('Probability', fontsize=10)    ax.set_title(f'{name}', fontsize=11, fontweight='bold')    ax.set_xticks(range(len(vocabulary)))    ax.set_xticklabels(vocabulary, rotation=45, ha='right', fontsize=9)    ax.grid(alpha=0.3)plt.tight_layout()plt.show()

## 7. Entropy Analysis

In [ ]:
# Compute Shannon entropy: H = -sum(p * log(p))def shannon_entropy(probs):    safe_probs = probs[probs > 0]    return -np.sum(safe_probs * np.log(safe_probs))entropy_temps = np.linspace(0.1, 2.0, 20)entropies = [shannon_entropy(apply_temperature(base_probs, t)) for t in entropy_temps]print("Shannon Entropy vs Temperature:")for temp, ent in zip(entropy_temps[::4], entropies[::4]):    print(f"  T={temp:.2f}: H={ent:.4f}")

In [ ]:
# Plot entropy vs temperaturefig, ax = plt.subplots(figsize=(8, 5))ax.plot(entropy_temps, entropies, marker='o', linewidth=2, markersize=6, color='#d62728')ax.set_xlabel('Temperature', fontsize=11, fontweight='bold')ax.set_ylabel('Shannon Entropy', fontsize=11, fontweight='bold')ax.set_title('Distribution Entropy vs Temperature', fontsize=12, fontweight='bold')ax.grid(alpha=0.3)ax.axhline(y=np.log(len(vocabulary)), color='gray', linestyle='--',            label=f'Max entropy (uniform): {np.log(len(vocabulary)):.3f}')ax.legend(fontsize=10)plt.tight_layout()plt.show()

## 8. Simulated Text Generation with Different Temperatures

In [ ]:
# Create a simple word frequency model from a small corpuscorpus = [    "The movie was great and wonderful",    "I think it was amazing and excellent",    "It was okay but not great",    "Terrible and poor quality",    "Great wonderful and amazing"]word_freq = defaultdict(int)for sentence in corpus:    for word in sentence.lower().split():        word_freq[word] += 1# Normalize to create probability distributionall_words = list(word_freq.keys())word_probs = np.array([word_freq[w] for w in all_words], dtype=float)word_probs = word_probs / word_probs.sum()print(f"Vocabulary size: {len(all_words)}")print(f"Top words: {sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:5]}")

In [ ]:
# Generate sample sentences at different temperaturesdef generate_sentence(n_words, temperature):    dist = apply_temperature(word_probs, temperature)    indices = np.random.choice(len(all_words), size=n_words, p=dist)    return ' '.join([all_words[i] for i in indices])np.random.seed(42)print("Generated Samples:\n")temps_for_generation = [0.2, 1.0, 2.0]for temp in temps_for_generation:    print(f"Temperature = {temp}:")    for i in range(5):        sentence = generate_sentence(8, temp)        print(f"  {i+1}. {sentence}")    print()

## 9. Structured Output Simulation

In [ ]:
# Sample product reviews for sentiment classificationsample_reviews = [    "This product is absolutely amazing and excellent!",    "Terrible quality, very poor experience.",    "It's okay, nothing special but works fine.",    "Wonderful product, great value for money!",    "Awful and disappointing, do not recommend."]# Keyword-based sentiment simulation (what an LLM would do)positive_keywords = ["amazing", "excellent", "wonderful", "great", "good"]negative_keywords = ["terrible", "poor", "awful", "disappointing", "decent"]def classify_sentiment(review):    words = review.lower().split()    pos_count = sum(1 for w in words if any(k in w for k in positive_keywords))    neg_count = sum(1 for w in words if any(k in w for k in negative_keywords))        if pos_count > neg_count:        sentiment = "positive"        score = 0.7 + 0.3 * min(pos_count / 3, 1.0)    elif neg_count > pos_count:        sentiment = "negative"        score = 0.3 * (1 - min(neg_count / 3, 1.0))    else:        sentiment = "neutral"        score = 0.5        topics = ["quality"] if any(k in review.lower() for k in ["quality", "product"]) else []    action = "investigate" if sentiment == "negative" else "none"        return {        "review": review,        "sentiment": sentiment,        "confidence": min(score, 1.0),        "topics": topics,        "action_needed": action    }results = [classify_sentiment(review) for review in sample_reviews]print("Structured Sentiment Classification Results:\n")for result in results:    print(json.dumps(result, indent=2))    print()

In [ ]:
# Display results as DataFrameresults_df = pd.DataFrame([    {        'Review': r['review'][:40] + '...' if len(r['review']) > 40 else r['review'],        'Sentiment': r['sentiment'],        'Confidence': f"{r['confidence']:.2f}",        'Topics': ', '.join(r['topics']) if r['topics'] else 'general',        'Action': r['action_needed']    }    for r in results])print("Sentiment Classification Summary:")print(results_df.to_string(index=False))

## 10. Parameter Recommendations by Use Case

In [ ]:
# Create recommendations tablerecommendations = pd.DataFrame([    {        'Use Case': 'Factual Q&A',        'Temperature': '0.2 - 0.5',        'Top-K': '30 - 50',        'Top-P': '0.8 - 0.95',        'Notes': 'Low T for consistency; high K/P for flexibility in valid answers'    },    {        'Use Case': 'Creative Writing',        'Temperature': '1.2 - 1.8',        'Top-K': '50 - 100',        'Top-P': '0.9 - 0.98',        'Notes': 'High T for diversity; relaxed K/P for expressive language'    },    {        'Use Case': 'Code Generation',        'Temperature': '0.3 - 0.7',        'Top-K': '40 - 80',        'Top-P': '0.85 - 0.95',        'Notes': 'Moderate T to balance correctness with variety'    },    {        'Use Case': 'Summarization',        'Temperature': '0.5 - 0.8',        'Top-K': '30 - 50',        'Top-P': '0.8 - 0.9',        'Notes': 'Moderate T to preserve meaning without excessive variation'    },    {        'Use Case': 'Dialogue',        'Temperature': '0.7 - 1.0',        'Top-K': '40 - 60',        'Top-P': '0.85 - 0.95',        'Notes': 'Balanced T for natural conversation; moderate K/P'    }])print("Recommended Parameter Ranges by Use Case:\n")print(recommendations.to_string(index=False))